Some loss Functions

Regression

Binary

Multi Label

Multi Class

TF-IDF

BAG of Words

KNN for text classification

Attention

In [ ]:

import numpy as np

In [ ]:
#Regression


class LinearRegression:

    def __init__(self, lr=0.01, epochs=20):
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # initialize weights
        self.W = np.random.randn(n_features, 1)
        self.b = 0

        y = y.reshape(-1, 1)

        for epoch in range(self.epochs):

            # prediction
            y_pred = X @ self.W + self.b

            # loss (MSE)
            loss = np.mean((y_pred - y) ** 2)

            # gradients
            dW = (2 / n_samples) * X.T @ (y_pred - y)
            db = (2 / n_samples) * np.sum(y_pred - y)

            # update
            self.W -= self.lr * dW
            self.b -= self.lr * db

            if epoch % 5 == 0:
                print("Epoch:", epoch, "Loss:", loss)

    def predict(self, X):
        return X @ self.W + self.b


In [ ]:
X = np.random.randn(100, 5)

# completely random target
y = np.random.randn(100, 1)

model = LinearRegression(lr=0.01, epochs=20)
model.fit(X, y)

preds = model.predict(X)


Epoch: 0 Loss: 5.215769182797957
Epoch: 5 Loss: 4.505398668780794
Epoch: 10 Loss: 3.9203795244451296
Epoch: 15 Loss: 3.437565252776834


In [ ]:
import numpy as np

class LinearRegression:

    def __init__(self, lr=0.01, epochs=20, l1=0.0, l2=0.0):
        self.lr = lr
        self.epochs = epochs
        self.l1 = l1  # L1 regularization strength
        self.l2 = l2  # L2 regularization strength

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # initialize weights
        self.W = np.random.randn(n_features, 1)
        self.b = 0

        y = y.reshape(-1, 1)

        for epoch in range(self.epochs):

            # 1. Prediction
            y_pred = X @ self.W + self.b

            # 2. Loss with L1 and L2 penalties
            mse = np.mean((y_pred - y) ** 2)
            l1_penalty = self.l1 * np.sum(np.abs(self.W))
            l2_penalty = self.l2 * np.sum(self.W**2)
            loss = mse + l1_penalty + l2_penalty

            # 3. Gradients
            # Base MSE gradient
            dW = (2 / n_samples) * X.T @ (y_pred - y)

            # Add Regularization Gradients
            # Derivative of |W| is sign(W); Derivative of W^2 is 2W
            dW += self.l1 * np.sign(self.W)
            dW += 2 * self.l2 * self.W

            # Bias gradient (usually not regularized)
            db = (2 / n_samples) * np.sum(y_pred - y)

            # 4. Update
            self.W -= self.lr * dW
            self.b -= self.lr * db

            if epoch % 5 == 0:
                print(f"Epoch: {epoch} | Loss: {loss:.4f}")

    def predict(self, X):
        return X @ self.W + self.b

# Example Usage:
# model = LinearRegression(lr=0.01, epochs=100, l1=0.01, l2=0.1)
# model.fit(X, y)

#Batch Gradient desent
def fit(self, X, y, batch_size=32):
    n_samples, n_features = X.shape
    self.W = np.random.randn(n_features, 1)
    self.b = 0
    y = y.reshape(-1, 1)

    for epoch in range(self.epochs):
        # 1. Shuffle at every epoch (Crucial for SGD!)
        indices = np.random.permutation(n_samples)
        X_sh = X[indices]
        y_sh = y[indices]

        # 2. Loop through mini-batches
        for i in range(0, n_samples, batch_size):
            X_i = X_sh[i : i + batch_size]
            y_i = y_sh[i : i + batch_size]
            m = X_i.shape[0] # Current batch size

            # Prediction on the batch only
            y_p = X_i @ self.W + self.b

            # Gradients based on the batch size 'm', not 'n_samples'
            dW = (2 / m) * X_i.T @ (y_p - y_i)
            dW += self.l1 * np.sign(self.W) + 2 * self.l2 * self.W
            db = (2 / m) * np.sum(y_p - y_i)

            # Update weights AFTER EVERY BATCH
            self.W -= self.lr * dW
            self.b -= self.lr * db



# import numpy as np

class LinearRegressionRMSprop:
    def __init__(self, lr=0.01, epochs=20, l1=0.0, l2=0.0, beta=0.9, epsilon=1e-8):
        self.lr = lr
        self.epochs = epochs
        self.l1 = l1
        self.l2 = l2
        self.beta = beta  # Decay rate (usually 0.9)
        self.epsilon = epsilon  # To prevent division by zero

        # State: Moving average of squared gradients
        self.s_W = None
        self.s_b = None

    def fit(self, X, y, batch_size=32):
        n_samples, n_features = X.shape
        self.W = np.random.randn(n_features, 1)
        self.b = 0
        y = y.reshape(-1, 1)

        # Initialize the squared gradient caches to zero
        self.s_W = np.zeros_like(self.W)
        self.s_b = 0

        for epoch in range(self.epochs):
            indices = np.random.permutation(n_samples)
            X_sh = X[indices]
            y_sh = y[indices]

            for i in range(0, n_samples, batch_size):
                X_i = X_sh[i : i + batch_size]
                y_i = y_sh[i : i + batch_size]
                m = X_i.shape[0]

                # 1. Gradients
                y_p = X_i @ self.W + self.b
                dW = (2 / m) * X_i.T @ (y_p - y_i)
                dW += self.l1 * np.sign(self.W) + 2 * self.l2 * self.W
                db = (2 / m) * np.sum(y_p - y_i)

                # 2. Update squared gradient caches (The RMSprop Core)
                # s = beta * s + (1 - beta) * (gradient^2)
                self.s_W = self.beta * self.s_W + (1 - self.beta) * (dW**2)
                self.s_b = self.beta * self.s_b + (1 - self.beta) * (db**2)

                # 3. Update Weights
                # We divide the gradient by the root of the cache
                self.W -= self.lr * dW / (np.sqrt(self.s_W) + self.epsilon)
                self.b -= self.lr * db / (np.sqrt(self.s_b) + self.epsilon)

            if epoch % 5 == 0:
                loss = np.mean((X @ self.W + self.b - y)**2)
                print(f"Epoch: {epoch} | Loss: {loss:.4f}")

    def predict(self, X):
        return X @ self.W + self.b




import numpy as np

class LinearRegressionMomentum:
    def __init__(self, lr=0.01, epochs=20, l1=0.0, l2=0.0, momentum=0.9):
        self.lr = lr
        self.epochs = epochs
        self.l1 = l1
        self.l2 = l2
        self.momentum = momentum

        # Velocity states (analogous to Adam's m_W and m_b)
        self.v_W = None
        self.v_b = None

    def fit(self, X, y, batch_size=32):
        n_samples, n_features = X.shape
        self.W = np.random.randn(n_features, 1)
        self.b = 0
        y = y.reshape(-1, 1)

        # Initialize velocities to zero
        self.v_W = np.zeros_like(self.W)
        self.v_b = 0

        for epoch in range(self.epochs):
            indices = np.random.permutation(n_samples)
            X_sh = X[indices]
            y_sh = y[indices]

            for i in range(0, n_samples, batch_size):
                X_i = X_sh[i : i + batch_size]
                y_i = y_sh[i : i + batch_size]
                m = X_i.shape[0]

                # 1. Standard Gradients
                y_p = X_i @ self.W + self.b
                dW = (2 / m) * X_i.T @ (y_p - y_i)
                dW += self.l1 * np.sign(self.W) + 2 * self.l2 * self.W
                db = (2 / m) * np.sum(y_p - y_i)

                # 2. Update Velocity (The Momentum Core)
                # v = gamma * v + lr * gradient
                self.v_W = self.momentum * self.v_W + self.lr * dW
                self.v_b = self.momentum * self.v_b + self.lr * db

                # 3. Update Weights
                self.W -= self.v_W
                self.b -= self.v_b

            if epoch % 5 == 0:
                loss = np.mean((X @ self.W + self.b - y)**2)
                print(f"Epoch: {epoch} | Loss: {loss:.4f}")

    def predict(self, X):
        return X @ self.W + self.b

In [ ]:
class Binary_Logistic:

    def __init__(self, lr=0.01, epochs=20, eps=1e-12):
        self.lr = lr
        self.epochs = epochs
        self.eps = eps

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def binary_loss(self, y, y_pred):
        return -np.mean(
            y * np.log(y_pred + self.eps) +
            (1 - y) * np.log(1 - y_pred + self.eps)
        )

    def fit(self, X, y):
        n_samples, n_features = X.shape

        self.W = np.random.randn(n_features, 1)
        self.b = 0

        y = y.reshape(-1, 1)

        for epoch in range(self.epochs):

            # forward
            logits = X @ self.W + self.b
            y_pred = self.sigmoid(logits)

            # loss
            loss = self.binary_loss(y, y_pred)

            # gradients
            dW = (1 / n_samples) * X.T @ (y_pred - y)
            db = (1 / n_samples) * np.sum(y_pred - y)

            # update
            self.W -= self.lr * dW
            self.b -= self.lr * db

            if epoch % 5 == 0:
                print("Epoch:", epoch, "Loss:", loss)

    def predict_proba(self, X):
        return self.sigmoid(X @ self.W + self.b)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

In [ ]:
X = np.random.randn(100, 5)

# random binary labels (0 or 1)
y = np.random.randint(0, 2, size=(100, 1))

model = Binary_Logistic(lr=0.01, epochs=20)
model.fit(X, y)

preds = model.predict(X)

Epoch: 0 Loss: 1.0582197849536887
Epoch: 5 Loss: 1.0529164106651727
Epoch: 10 Loss: 1.0476647352324342
Epoch: 15 Loss: 1.0424647546594004


In [ ]:
import numpy as np

class MultiClass_Logistic:

    def __init__(self, lr=0.01, epochs=20, eps=1e-12):
        self.lr = lr
        self.epochs = epochs
        self.eps = eps

    def softmax(self, z):
        z = z - np.max(z, axis=1, keepdims=True)  # stability
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def loss(self, y, y_pred):

        return -np.mean(np.sum(y * np.log(y_pred + self.eps), axis=1))


    def fit(self, X, y):
        n_samples, n_features = X.shape
        n_classes = y.shape[1]

        # initialize parameters
        self.W = np.random.randn(n_features, n_classes)
        self.b = np.zeros((1, n_classes))

        for epoch in range(self.epochs):

            # -------- forward pass --------
            logits = X @ self.W + self.b
            y_pred = self.softmax(logits)

            # -------- loss --------
            loss = self.loss(y, y_pred)

            # -------- gradients --------
            dW = (1 / n_samples) * X.T @ (y_pred - y)
            db = (1 / n_samples) * np.sum(y_pred - y, axis=0, keepdims=True)

            # -------- update --------
            self.W -= self.lr * dW
            self.b -= self.lr * db

            if epoch % 5 == 0:
                print("Epoch:", epoch, "Loss:", loss)

    def predict(self, X):
        logits = X @ self.W + self.b
        probs = self.softmax(logits)
        return np.argmax(probs, axis=1)


In [ ]:
X = np.random.randn(100, 5)

# random class labels: 0, 1, 2
labels = np.random.randint(0, 3, size=100)

# one-hot encoding
y = np.zeros((100, 3))
y[np.arange(100), labels] = 1

# train
model = MultiClass_Logistic(lr=0.01, epochs=30)
model.fit(X, y)

# predict
preds = model.predict(X)
print(preds[:10])

Epoch: 0 Loss: 2.2736466402756603
Epoch: 5 Loss: 2.255380632851171
Epoch: 10 Loss: 2.2372795406329855
Epoch: 15 Loss: 2.2193435070981407
Epoch: 20 Loss: 2.2015726375148033
Epoch: 25 Loss: 2.183966999496268
[0 0 1 1 2 0 0 1 1 0]


In [ ]:
import numpy as np

class MultiLabel_Logistic:

    def __init__(self, lr=0.01, epochs=20, eps=1e-12):
        self.lr = lr
        self.epochs = epochs
        self.eps = eps

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def loss(self, y, y_pred):
        return -np.mean(
            y * np.log(y_pred + self.eps) +
            (1 - y) * np.log(1 - y_pred + self.eps)
        )

    #     return -np.mean(
    #     np.sum(
    #         y * np.log(y_pred + self.eps)
    #         + (1 - y) * np.log(1 - y_pred + self.eps),
    #         axis=1
    #     )
    # )

    def fit(self, X, y):
        n_samples, n_features = X.shape
        n_labels = y.shape[1]

        # initialize parameters
        self.W = np.random.randn(n_features, n_labels)
        self.b = np.zeros((1, n_labels))

        for epoch in range(self.epochs):

            # -------- forward --------
            logits = X @ self.W + self.b
            y_pred = self.sigmoid(logits)

            # -------- loss --------
            loss = self.loss(y, y_pred)

            # -------- gradients --------
            dW = (1 / n_samples) * X.T @ (y_pred - y)
            db = (1 / n_samples) * np.sum(y_pred - y, axis=0, keepdims=True)

            # -------- update --------
            self.W -= self.lr * dW
            self.b -= self.lr * db

            if epoch % 5 == 0:
                print("Epoch:", epoch, "Loss:", loss)

    def predict(self, X, threshold=0.5):
        probs = self.sigmoid(X @ self.W + self.b)
        return (probs >= threshold).astype(int)


In [ ]:
X = np.random.randn(100, 5)

# multilabel targets (each sample can have multiple 1s)
y = np.random.randint(0, 2, size=(100, 3))

model = MultiLabel_Logistic(lr=0.01, epochs=30)
model.fit(X, y)

preds = model.predict(X)
print(preds[:5])

Epoch: 0 Loss: 1.1238068807496593
Epoch: 5 Loss: 1.1177164201630891
Epoch: 10 Loss: 1.1116750102040576
Epoch: 15 Loss: 1.1056828138967876
Epoch: 20 Loss: 1.0997399863807178
Epoch: 25 Loss: 1.0938466747396558
[[0 0 1]
 [0 1 0]
 [0 0 1]
 [1 1 0]
 [0 1 0]]


In [ ]:
import numpy as np

def softmax(x, axis=-1):
    # numerical stability
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def self_attention(X, Wq, Wk, Wv):
    """
    X  : (n_tokens, d_model)
    Wq : (d_model, d_k)
    Wk : (d_model, d_k)
    Wv : (d_model, d_v)

    returns:
    output : (n_tokens, d_v)
    """

    # 1. Linear projections
    Q = X @ Wq      # (n_tokens, d_k)
    K = X @ Wk      # (n_tokens, d_k)
    V = X @ Wv      # (n_tokens, d_v)

    # 2. Scaled dot-product attention
    d_k = Q.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d_k)   # (n_tokens, n_tokens)

    # 3. Softmax
    weights = softmax(scores, axis=-1)  # (n_tokens, n_tokens)

    # 4. Weighted sum
    output = weights @ V                # (n_tokens, d_v)

    return output

In [ ]:
np.random.seed(0)

n_tokens = 4
d_model = 8
d_k = 4
d_v = 4

X = np.random.randn(n_tokens, d_model)
Wq = np.random.randn(d_model, d_k)
Wk = np.random.randn(d_model, d_k)
Wv = np.random.randn(d_model, d_v)

out = self_attention(X, Wq, Wk, Wv)
print(out.shape)   # (4, 4)

In [ ]:
def masked_self_attention(X, Wq, Wk, Wv):
    """
    X : (n_tokens, d_model)
    """

    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv

    d_k = Q.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d_k)

    # Causal mask (prevent attending to future tokens)
    n = scores.shape[0]
    mask = np.triu(np.ones((n, n)), k=1)
    scores = scores - 1e9 * mask

    weights = softmax(scores, axis=-1)
    output = weights @ V

    return output

In [ ]:
def cross_attention(X_dec, X_enc, Wq, Wk, Wv):
    """
    X_dec : (n_dec_tokens, d_model)
    X_enc : (n_enc_tokens, d_model)
    """

    Q = X_dec @ Wq
    K = X_enc @ Wk
    V = X_enc @ Wv

    d_k = Q.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d_k)

    weights = softmax(scores, axis=-1)
    output = weights @ V

    return output

In [ ]:
a = np.zeros((1, 5, 1, 3))

b = np.squeeze(a, axis=2)
print(b.shape)
# (1, 5, 3)

In [ ]:
def classification_metrics(y_true, y_pred):
    tp = fp = fn = 0

    for t, p in zip(y_true, y_pred):
        if t == 1 and p == 1:
            tp += 1
        elif t == 0 and p == 1:
            fp += 1
        elif t == 1 and p == 0:
            fn += 1

    # Precision
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0

    # Recall
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0

    # F1 Score
    if (precision + recall) != 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0

    return precision, recall, f1

In [ ]:
import math
from collections import Counter

def compute_tfidf(docs):
    N = len(docs)

    # Tokenize
    docs_tokens = [doc.lower().split() for doc in docs]

    # Document Frequency (DF)
    df = Counter()
    for doc in docs_tokens:
        for word in set(doc):
            df[word] += 1

    tfidf = []

    for doc in docs_tokens:
        tf = Counter(doc)
        doc_len = len(doc)
        doc_tfidf = {}

        for word in tf:
            tf_val = tf[word] / doc_len
            idf_val = math.log(N / (1 + df[word]))  # smoothing
            doc_tfidf[word] = tf_val * idf_val

        tfidf.append(doc_tfidf)

    return tfidf


docs = [
    "I love machine learning",
    "machine learning is fun",
    "I love coding"
]

result = compute_tfidf(docs)

for i, doc in enumerate(result):
    print(f"Doc {i+1}:", doc)

In [ ]:
import numpy as np

def stable_softmax(x, axis=-1):
    # Subtracting the max prevents exponential overflow
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def single_head_attention(X, W_q, W_k, W_v, mask=None):
    """
    Computes Causal Self-Attention for a single head.

    Shapes:
        X: (batch_size, seq_len, d_model)
        W_q, W_k, W_v: (d_model, head_size)
    """
    # 1. Project input matrix X into Q, K, V space
    # (B, T, d_model) @ (d_model, head_size) -> (B, T, head_size) * (B, head_size, T) = (B,T,T)
    Q = np.matmul(X, W_q)
    K = np.matmul(X, W_k)
    V = np.matmul(X, W_v)

    head_size = K.shape[-1]

    # 2. Transpose K's last two dimensions: (B, T, head_size) -> (B, head_size, T)
    K_T = np.transpose(K, axes=(0, 2, 1))

    # 3. Calculate raw attention scores
    # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
    scores = np.matmul(Q, K_T) / np.sqrt(head_size)

    # 4. Apply Causal Masking if provided
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)

    # 5. Compute attention weights (probabilities)
    attention_weights = stable_softmax(scores, axis=-1)

    # 6. Weight the Values matrix
    # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
    output = np.matmul(attention_weights, V)

    return output, attention_weights


# --- Execution Example ---
if __name__ == "__main__":
    batch_size = 1
    seq_len = 3
    d_model = 8
    head_size = 4    # Dynamic size of this isolated head (d_k)

    np.random.seed(42)

    # Initialize inputs and weights
    X = np.random.randn(batch_size, seq_len, d_model)
    W_q = np.random.randn(d_model, head_size)
    W_k = np.random.randn(d_model, head_size)
    W_v = np.random.randn(d_model, head_size)

    # Lower triangular causal mask
    causal_mask = np.tril(np.ones((seq_len, seq_len)))

    # Run pipeline
    context_vector, weights = single_head_attention(X, W_q, W_k, W_v, mask=causal_mask)

    print("Attention Weights (Probabilities):\n", np.round(weights[0], 4))
    print("\nOutput Shape:", context_vector.shape)



import numpy as np

# ----------------------------------------------------------
# Stable Softmax
# ----------------------------------------------------------

def stable_softmax(x, axis=-1):
    """
    x : (..., n)

    Returns:
        (..., n)
    """

    x = x - np.max(x, axis=axis, keepdims=True)
    exp = np.exp(x)

    return exp / np.sum(exp, axis=axis, keepdims=True)


# ----------------------------------------------------------
# Single Head Self Attention
# ----------------------------------------------------------

def single_head_attention(X, W_q, W_k, W_v, mask=None):
    """
    Parameters
    ----------
    X   : (B, T, d_model)

    W_q : (d_model, head_size)
    W_k : (d_model, head_size)
    W_v : (d_model, head_size)

    mask: (T, T)

    Returns
    -------
    output             : (B, T, head_size)
    attention_weights  : (B, T, T)
    """

    # ---------------------------------------
    # Step 1 : Compute Query, Key and Value
    # ---------------------------------------

    # (B,T,d_model) @ (d_model,head_size)
    # ->
    # (B,T,head_size)

    Q = X @ W_q
    K = X @ W_k
    V = X @ W_v


    # ---------------------------------------
    # Step 2 : Transpose Key
    # ---------------------------------------

    # (B,T,head_size)
    # ->
    # (B,head_size,T)

    K_T = np.transpose(K, (0, 2, 1))


    # ---------------------------------------
    # Step 3 : Attention Scores
    # ---------------------------------------

    # (B,T,head_size)
    # @
    # (B,head_size,T)
    #
    # ->
    #
    # (B,T,T)

    dk = K.shape[-1]

    scores = (Q @ K_T) / np.sqrt(dk)


    # ---------------------------------------
    # Step 4 : Apply Mask
    # ---------------------------------------

    # scores : (B,T,T)
    # mask   : (T,T)

    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)


    # ---------------------------------------
    # Step 5 : Softmax
    # ---------------------------------------

    # (B,T,T)

    attention_weights = stable_softmax(scores, axis=-1)


    # ---------------------------------------
    # Step 6 : Weighted Sum of Values
    # ---------------------------------------

    # (B,T,T)
    # @
    # (B,T,head_size)
    #
    # ->
    #
    # (B,T,head_size)

    output = attention_weights @ V

    return output, attention_weights



import numpy as np

def multi_head_attention(
    X,
    W_q_list,
    W_k_list,
    W_v_list,
    W_o,
    mask=None
):
    """
    Parameters
    ----------

    X : (B,T,d_model)

    W_q_list : List[(d_model,head_size)]
    W_k_list : List[(d_model,head_size)]
    W_v_list : List[(d_model,head_size)]

    W_o : (num_heads*head_size, d_model)

    Returns
    -------

    output : (B,T,d_model)
    """

    head_outputs = []

    # ---------------------------------------
    # Compute every head independently
    # ---------------------------------------

    for W_q, W_k, W_v in zip(W_q_list, W_k_list, W_v_list):

        # out : (B,T,head_size)

        out, _ = single_head_attention(
            X,
            W_q,
            W_k,
            W_v,
            mask
        )

        head_outputs.append(out)


    # ---------------------------------------
    # Concatenate all heads
    # ---------------------------------------

    # Example:
    #
    # Head1 : (B,T,64)
    # Head2 : (B,T,64)
    # Head3 : (B,T,64)
    # Head4 : (B,T,64)
    #
    # ->
    #
    # (B,T,256)

    concat = np.concatenate(head_outputs, axis=-1)


    # ---------------------------------------
    # Final Output Projection
    # ---------------------------------------

    # (B,T,num_heads*head_size)
    #
    # @
    #
    # (num_heads*head_size,d_model)
    #
    # ->
    #
    # (B,T,d_model)

    output = concat @ W_o

    return output

In [ ]:
import numpy as np


# ==========================================================
# Stable Softmax
# ==========================================================

def stable_softmax(x, axis=-1):

    # Subtract max for numerical stability
    x = x - np.max(x, axis=axis, keepdims=True)

    # Exponential
    exp_x = np.exp(x)

    # Normalize
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


# ==========================================================
# Single Head Self-Attention
# ==========================================================

def single_head_attention(X, W_q, W_k, W_v, mask=None):

    # X   : (B, T, d_model)
    # W_q : (d_model, head_size)
    # W_k : (d_model, head_size)
    # W_v : (d_model, head_size)

    # ------------------------------------------------------
    # Step 1: Compute Q, K, V
    # ------------------------------------------------------

    Q = X @ W_q
    K = X @ W_k
    V = X @ W_v

    # Q : (B, T, head_size)
    # K : (B, T, head_size)
    # V : (B, T, head_size)


    # ------------------------------------------------------
    # Step 2: Transpose K
    # ------------------------------------------------------

    # (B, T, head_size)
    # ->
    # (B, head_size, T)

    K_T = np.transpose(K, (0, 2, 1))


    # ------------------------------------------------------
    # Step 3: Compute Attention Scores
    # ------------------------------------------------------

    d_k = K.shape[-1]

    # (B, T, head_size)
    # @
    # (B, head_size, T)
    # ->
    # (B, T, T)

    scores = (Q @ K_T) / np.sqrt(d_k)


    # ------------------------------------------------------
    # Step 4: Apply Mask
    # ------------------------------------------------------

    if mask is not None:

        scores = np.where(
            mask == 0,
            -1e9,
            scores
        )


    # ------------------------------------------------------
    # Step 5: Softmax
    # ------------------------------------------------------

    attention_weights = stable_softmax(
        scores,
        axis=-1
    )


    # ------------------------------------------------------
    # Step 6: Weighted Sum of Values
    # ------------------------------------------------------

    # (B, T, T)
    # @
    # (B, T, head_size)
    # ->
    # (B, T, head_size)

    output = attention_weights @ V


    return output, attention_weights


# ==========================================================
# Multi-Head Attention
# ==========================================================

def multi_head_attention(
    X,
    W_q_list,
    W_k_list,
    W_v_list,
    W_o,
    mask=None
):

    # ------------------------------------------------------
    # Step 1: Store outputs of each head
    # ------------------------------------------------------

    head_outputs = []


    # ------------------------------------------------------
    # Step 2: Compute each head
    # ------------------------------------------------------

    for W_q, W_k, W_v in zip(
        W_q_list,
        W_k_list,
        W_v_list
    ):

        # Compute one attention head

        out, _ = single_head_attention(
            X,
            W_q,
            W_k,
            W_v,
            mask
        )

        head_outputs.append(out)


    # ------------------------------------------------------
    # Step 3: Concatenate heads
    # ------------------------------------------------------

    # Each head:
    # (B, T, head_size)
    #
    # num_heads heads
    #
    # ->
    #
    # (B, T, num_heads * head_size)

    concat = np.concatenate(
        head_outputs,
        axis=-1
    )


    # ------------------------------------------------------
    # Step 4: Final Output Projection
    # ------------------------------------------------------

    # (B, T, num_heads * head_size)
    #
    # @
    #
    # (num_heads * head_size, d_model)
    #
    # ->
    #
    # (B, T, d_model)

    output = concat @ W_o


    return output


# ==========================================================
# Execution Example
# ==========================================================

if __name__ == "__main__":

    # ------------------------------------------------------
    # Dimensions
    # ------------------------------------------------------

    batch_size = 1
    seq_len = 3
    d_model = 8

    num_heads = 2
    head_size = 4


    # ------------------------------------------------------
    # Input
    # ------------------------------------------------------

    np.random.seed(42)

    X = np.random.randn(
        batch_size,
        seq_len,
        d_model
    )


    # ------------------------------------------------------
    # Initialize weights for each head
    # ------------------------------------------------------

    W_q_list = [
        np.random.randn(d_model, head_size)
        for _ in range(num_heads)
    ]

    W_k_list = [
        np.random.randn(d_model, head_size)
        for _ in range(num_heads)
    ]

    W_v_list = [
        np.random.randn(d_model, head_size)
        for _ in range(num_heads)
    ]


    # ------------------------------------------------------
    # Output projection
    # ------------------------------------------------------

    W_o = np.random.randn(
        num_heads * head_size,
        d_model
    )


    # ------------------------------------------------------
    # Causal Mask
    # ------------------------------------------------------

    causal_mask = np.tril(
        np.ones(
            (seq_len, seq_len)
        )
    )


    # ------------------------------------------------------
    # Run Multi-Head Attention
    # ------------------------------------------------------

    output = multi_head_attention(
        X,
        W_q_list,
        W_k_list,
        W_v_list,
        W_o,
        mask=causal_mask
    )


    # ------------------------------------------------------
    # Print
    # ------------------------------------------------------

    print("Input Shape:", X.shape)

    print("Output Shape:", output.shape)

    print("\nOutput:")
    print(output)

In [ ]:
```python
import torch
import torch.nn as nn


# ----------------------------------------------------------
# Stable Softmax
# ----------------------------------------------------------

def stable_softmax(x, dim=-1):
    """
    x : (..., n)

    Returns
    -------
    (..., n)
    """

    x = x - torch.max(x, dim=dim, keepdim=True).values
    exp = torch.exp(x)

    return exp / torch.sum(exp, dim=dim, keepdim=True)


# ----------------------------------------------------------
# Single Head Self Attention
# ----------------------------------------------------------

class SingleHeadAttention(nn.Module):

    def __init__(self, d_model, head_size):
        super().__init__()

        # (d_model -> head_size)

        self.W_q = nn.Linear(d_model, head_size, bias=False)
        self.W_k = nn.Linear(d_model, head_size, bias=False)
        self.W_v = nn.Linear(d_model, head_size, bias=False)

    def forward(self, X, mask=None):
        """
        X : (B,T,d_model)
        """

        # ---------------------------------------
        # Compute Query, Key, Value
        #
        # (B,T,d_model)
        #
        # ->
        #
        # (B,T,head_size)
        # ---------------------------------------

        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        # ---------------------------------------
        # Transpose Key
        #
        # (B,T,head_size)
        #
        # ->
        #
        # (B,head_size,T)
        # ---------------------------------------

        K_T = K.transpose(1, 2)

        # ---------------------------------------
        # Attention Scores
        #
        # (B,T,head_size)
        #
        # @
        #
        # (B,head_size,T)
        #
        # ->
        #
        # (B,T,T)
        # ---------------------------------------

        head_size = K.size(-1)

        scores = (Q @ K_T) / (head_size ** 0.5)

        # ---------------------------------------
        # Causal Mask
        # ---------------------------------------

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        # ---------------------------------------
        # Softmax
        # ---------------------------------------

        attention_weights = stable_softmax(scores, dim=-1)

        # ---------------------------------------
        # Weighted Sum
        #
        # (B,T,T)
        #
        # @
        #
        # (B,T,head_size)
        #
        # ->
        #
        # (B,T,head_size)
        # ---------------------------------------

        output = attention_weights @ V

        return output, attention_weights


# ----------------------------------------------------------
# Multi Head Attention
# ----------------------------------------------------------

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):

        super().__init__()

        assert d_model % num_heads == 0

        head_size = d_model // num_heads

        self.heads = nn.ModuleList(

            [
                SingleHeadAttention(
                    d_model,
                    head_size
                )

                for _ in range(num_heads)
            ]
        )

        self.W_o = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

    def forward(self, X, mask=None):
        """
        X : (B,T,d_model)
        """

        head_outputs = []

        # ---------------------------------------
        # Run Every Head
        # ---------------------------------------

        for head in self.heads:

            out, _ = head(X, mask)

            head_outputs.append(out)

        # ---------------------------------------
        # Concatenate Heads
        #
        # Head1 : (B,T,head_size)
        # Head2 : (B,T,head_size)
        #
        # ->
        #
        # (B,T,d_model)
        # ---------------------------------------

        concat = torch.cat(head_outputs, dim=-1)

        # ---------------------------------------
        # Output Projection
        #
        # (B,T,d_model)
        #
        # ->
        #
        # (B,T,d_model)
        # ---------------------------------------

        output = self.W_o(concat)

        return output


# ----------------------------------------------------------
# Example
# ----------------------------------------------------------

if __name__ == "__main__":

    batch_size = 2
    seq_len = 5
    d_model = 8
    num_heads = 2

    torch.manual_seed(42)

    X = torch.randn(batch_size, seq_len, d_model)

    causal_mask = torch.tril(torch.ones(seq_len, seq_len))

    mha = MultiHeadAttention(
        d_model=d_model,
        num_heads=num_heads
    )

    output = mha(
        X,
        causal_mask
    )

    print(output.shape)
```


In [ ]:
import numpy as np


# ==========================================================
# K-Nearest Neighbors
# ==========================================================

class KNN:

    def __init__(self, k=3, task="classification"):

        self.k = k
        self.task = task

        self.X_train = None
        self.y_train = None


    # ======================================================
    # Fit
    # ======================================================

    def fit(self, X, y):

        # TODO: store training data

        self.X_train = X
        self.y_train = y

        return self


    # ======================================================
    # Calculate Distance
    # ======================================================

    def distance(self, x1, x2):

        # Euclidean distance
        #
        # sqrt(
        #     (x1_1 - x2_1)^2 +
        #     (x1_2 - x2_2)^2 +
        #     ...
        # )

        # TODO: calculate Euclidean distance

        return np.sqrt(
            np.sum((x1 - x2) ** 2)
        )


    # ======================================================
    # Predict One Sample
    # ======================================================

    def predict_one(self, x):

        # --------------------------------------------------
        # Step 1: Calculate distance from x to every
        #         training example
        # --------------------------------------------------

        distances = []

        for i in range(len(self.X_train)):

            # TODO: calculate distance

            dist = self.distance(
                x,
                self.X_train[i]
            )

            distances.append(dist)


        # --------------------------------------------------
        # Step 2: Find K nearest points
        # --------------------------------------------------

        distances = np.array(distances)

        # TODO: get indices of K smallest distances

        nearest_indices = np.argsort(
            distances
        )[:self.k]


        # --------------------------------------------------
        # Step 3: Get labels of K nearest points
        # --------------------------------------------------

        nearest_labels = self.y_train[
            nearest_indices
        ]


        # --------------------------------------------------
        # Step 4: Make prediction
        # --------------------------------------------------

        if self.task == "classification":

            # TODO:
            # Find most frequent class

            values, counts = np.unique(
                nearest_labels,
                return_counts=True
            )

            prediction = values[
                np.argmax(counts)
            ]

        else:

            # Regression
            #
            # TODO: take mean of nearest labels

            prediction = np.mean(
                nearest_labels
            )


        return prediction


    # ======================================================
    # Predict
    # ======================================================

    def predict(self, X):

        predictions = []

        # TODO: predict each sample

        for x in X:

            predictions.append(
                self.predict_one(x)
            )

        return np.array(predictions)


# ==========================================================
# Execution Example - Classification
# ==========================================================

if __name__ == "__main__":

    np.random.seed(42)


    # ------------------------------------------------------
    # Create Training Data
    # ------------------------------------------------------

    X_train = np.array([
        [1, 2],
        [2, 3],
        [3, 3],
        [6, 5],
        [7, 7],
        [8, 6]
    ])


    # Class labels

    y_train = np.array([
        0,
        0,
        0,
        1,
        1,
        1
    ])


    # ------------------------------------------------------
    # Create Test Data
    # ------------------------------------------------------

    X_test = np.array([
        [2, 2],
        [7, 6],
        [5, 5]
    ])


    # ------------------------------------------------------
    # Train Model
    # ------------------------------------------------------

    model = KNN(
        k=3,
        task="classification"
    )

    model.fit(
        X_train,
        y_train
    )


    # ------------------------------------------------------
    # Prediction
    # ------------------------------------------------------

    predictions = model.predict(
        X_test
    )


    # ------------------------------------------------------
    # Print Results
    # ------------------------------------------------------

    print("Predictions:")
    print(predictions)

In [ ]:
import numpy as np


# ==========================================================
# PCA
# ==========================================================

class PCA:

    def __init__(self, n_components):

        self.n_components = n_components

        self.mean = None
        self.components = None
        self.eigenvalues = None


    # ======================================================
    # Fit
    # ======================================================

    def fit(self, X):

        # --------------------------------------------------
        # Step 1: Calculate mean
        # --------------------------------------------------

        self.mean = np.mean(
            X,
            axis=0
        )


        # --------------------------------------------------
        # Step 2: Center the data
        # --------------------------------------------------

        X_centered = X - self.mean


        # --------------------------------------------------
        # Step 3: Calculate covariance matrix
        # --------------------------------------------------

        cov_matrix = np.cov(
            X_centered,
            rowvar=False
        )


        # --------------------------------------------------
        # Step 4: Calculate eigenvalues/eigenvectors
        # --------------------------------------------------

        eigenvalues, eigenvectors = np.linalg.eigh(
            cov_matrix
        )


        # --------------------------------------------------
        # Step 5: Sort eigenvalues in descending order
        # --------------------------------------------------

        indices = np.argsort(
            eigenvalues
        )[::-1]


        eigenvalues = eigenvalues[indices]

        eigenvectors = eigenvectors[:, indices]


        # --------------------------------------------------
        # Step 6: Select top K components
        # --------------------------------------------------

        self.eigenvalues = eigenvalues[
            :self.n_components
        ]

        self.components = eigenvectors[
            :, :self.n_components
        ]


        return self


    # ======================================================
    # Transform
    # ======================================================

    def transform(self, X):

        # --------------------------------------------------
        # Step 1: Center new data using training mean
        # --------------------------------------------------

        X_centered = X - self.mean


        # --------------------------------------------------
        # Step 2: Project onto principal components
        # --------------------------------------------------

        return X_centered @ self.components


    # ======================================================
    # Fit + Transform
    # ======================================================

    def fit_transform(self, X):

        self.fit(X)

        return self.transform(X)


# ==========================================================
# Execution Example
# ==========================================================

if __name__ == "__main__":

    np.random.seed(42)


    # ------------------------------------------------------
    # Create Dataset
    # ------------------------------------------------------

    X = np.random.randn(
        100,
        5
    )


    print("Original shape:", X.shape)


    # ------------------------------------------------------
    # Create PCA Model
    # ------------------------------------------------------

    model = PCA(
        n_components=2
    )


    # ------------------------------------------------------
    # Fit PCA
    # ------------------------------------------------------

    model.fit(X)


    # ------------------------------------------------------
    # Transform Data
    # ------------------------------------------------------

    X_reduced = model.transform(X)


    # ------------------------------------------------------
    # Print Results
    # ------------------------------------------------------

    print("Reduced shape:", X_reduced.shape)

    print("\nPrincipal Components:")

    print(model.components)

    print("\nEigenvalues:")

    print(model.eigenvalues)


    # ------------------------------------------------------
    # Alternative: fit_transform()
    # ------------------------------------------------------

    X_reduced_2 = model.fit_transform(X)

    print(
        "\nFit-transform shape:",
        X_reduced_2.shape
    )

In [ ]:
import numpy as np


class KMeans:

    def __init__(self, k=3, epochs=100):

        self.k = k
        self.epochs = epochs
        self.centroids = None


    def fit(self, X):

        n_samples = X.shape[0]

        # ------------------------------------------
        # Initialize centroids
        # ------------------------------------------

        indices = np.random.choice(
            n_samples,
            self.k,
            replace=False
        )

        self.centroids = X[indices]


        # ------------------------------------------
        # Training
        # ------------------------------------------

        for epoch in range(self.epochs):

            # --------------------------------------
            # Assign points to clusters
            # --------------------------------------

            labels = []

            for x in X:

                distances = []

                for centroid in self.centroids:

                    distance = np.sqrt(
                        np.sum((x - centroid) ** 2)
                    )

                    distances.append(distance)

                label = np.argmin(distances)

                labels.append(label)


            labels = np.array(labels)


            # --------------------------------------
            # Recalculate centroids
            # --------------------------------------

            new_centroids = []

            for i in range(self.k):

                cluster_points = X[
                    labels == i
                ]

                centroid = np.mean(
                    cluster_points,
                    axis=0
                )

                new_centroids.append(centroid)


            new_centroids = np.array(
                new_centroids
            )


            # --------------------------------------
            # Check convergence
            # --------------------------------------

            if np.allclose(
                self.centroids,
                new_centroids
            ):
                break


            self.centroids = new_centroids


        self.labels = labels

        return self


    def predict(self, X):

        predictions = []

        for x in X:

            distances = []

            for centroid in self.centroids:

                distance = np.sqrt(
                    np.sum((x - centroid) ** 2)
                )

                distances.append(distance)

            predictions.append(
                np.argmin(distances)
            )

        return np.array(predictions)


# ==========================================================
# Execution
# ==========================================================

if __name__ == "__main__":

    # ------------------------------------------
    # Create X
    # ------------------------------------------

    X = np.array([
        [1, 2],
        [2, 3],
        [1, 3],
        [8, 8],
        [9, 8],
        [8, 9],
        [4, 10],
        [5, 10],
        [4, 11]
    ])


    # ------------------------------------------
    # Create Model
    # ------------------------------------------

    model = KMeans(
        k=3,
        epochs=100
    )


    # ------------------------------------------
    # Fit Model
    # ------------------------------------------

    model.fit(X)


    # ------------------------------------------
    # Prediction
    # ------------------------------------------

    predictions = model.predict(X)


    # ------------------------------------------
    # Print Results
    # ------------------------------------------

    print("Centroids:")
    print(model.centroids)

    print("\nCluster assignments:")
    print(predictions)

In [ ]:
import numpy as np


# ==========================================================
# Decision Tree Classifier
# ==========================================================

class DecisionTree:

    def __init__(
        self,
        max_depth=5,
        min_samples_split=2
    ):

        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None


    # ======================================================
    # Gini Impurity
    # ======================================================

    def gini(self, y):

        # TODO: get unique classes and counts

        classes, counts = np.unique(
            y,
            return_counts=True
        )


        # TODO: calculate probabilities

        probabilities = counts / len(y)


        # TODO: calculate Gini

        return 1 - np.sum(
            probabilities ** 2
        )




    # ======================================================
    # Find Best Split
    # ======================================================

    def best_split(self, X, y):

        n_samples, n_features = X.shape

        best_feature = None
        best_threshold = None
        best_gini = float("inf")


        # --------------------------------------------------
        # Try every feature
        # --------------------------------------------------

        for feature in range(n_features):

            # Get possible thresholds

            thresholds = np.unique(
                X[:, feature]
            )


            # --------------------------------------------------
            # Try every threshold
            # --------------------------------------------------

            for threshold in thresholds:

                # Split data

                left_mask = (
                    X[:, feature] <= threshold
                )

                right_mask = (
                    X[:, feature] > threshold
                )


                # Ignore invalid splits

                if (
                    np.sum(left_mask) == 0
                    or
                    np.sum(right_mask) == 0
                ):
                    continue


                y_left = y[left_mask]

                y_right = y[right_mask]


                # --------------------------------------------------
                # Calculate weighted Gini
                # --------------------------------------------------

                n_left = len(y_left)
                n_right = len(y_right)

                n_total = len(y)


                weighted_gini = (
                    (n_left / n_total) *
                    self.gini(y_left)
                    +
                    (n_right / n_total) *
                    self.gini(y_right)
                )


                # --------------------------------------------------
                # Keep best split
                # --------------------------------------------------

                if weighted_gini < best_gini:

                    best_gini = weighted_gini

                    best_feature = feature

                    best_threshold = threshold


        return (
            best_feature,
            best_threshold
        )


    # ======================================================
    # Build Tree Recursively
    # ======================================================

    def build_tree(
        self,
        X,
        y,
        depth=0
    ):

        # --------------------------------------------------
        # Stopping conditions
        # --------------------------------------------------

        # If all labels are the same

        if len(np.unique(y)) == 1:

            return {
                "leaf": True,
                "prediction": y[0]
            }


        # Maximum depth reached

        if depth >= self.max_depth:

            return {
                "leaf": True,
                "prediction": self.majority_class(y)
            }


        # Too few samples to split

        if len(y) < self.min_samples_split:

            return {
                "leaf": True,
                "prediction": self.majority_class(y)
            }


        # --------------------------------------------------
        # Find best split
        # --------------------------------------------------

        feature, threshold = self.best_split(
            X,
            y
        )


        # If no valid split exists

        if feature is None:

            return {
                "leaf": True,
                "prediction": self.majority_class(y)
            }


        # --------------------------------------------------
        # Split data
        # --------------------------------------------------

        left_mask = (
            X[:, feature] <= threshold
        )

        right_mask = (
            X[:, feature] > threshold
        )


        X_left = X[left_mask]

        y_left = y[left_mask]

        X_right = X[right_mask]

        y_right = y[right_mask]


        # --------------------------------------------------
        # Recursively build left tree
        # --------------------------------------------------

        left_tree = self.build_tree(
            X_left,
            y_left,
            depth + 1
        )


        # --------------------------------------------------
        # Recursively build right tree
        # --------------------------------------------------

        right_tree = self.build_tree(
            X_right,
            y_right,
            depth + 1
        )


        # --------------------------------------------------
        # Return current node
        # --------------------------------------------------

        return {
            "leaf": False,
            "feature": feature,
            "threshold": threshold,
            "left": left_tree,
            "right": right_tree
        }


    # ======================================================
    # Majority Class
    # ======================================================

    def majority_class(self, y):

        classes, counts = np.unique(
            y,
            return_counts=True
        )

        return classes[
            np.argmax(counts)
        ]


    # ======================================================
    # Fit
    # ======================================================

    def fit(self, X, y):

        self.tree = self.build_tree(
            X,
            y
        )

        return self


    # ======================================================
    # Predict One Sample
    # ======================================================

    def predict_one(
        self,
        x,
        node
    ):

        # --------------------------------------------------
        # If leaf, return prediction
        # --------------------------------------------------

        if node["leaf"]:

            return node["prediction"]


        # --------------------------------------------------
        # Decide left or right
        # --------------------------------------------------

        if x[node["feature"]] <= node["threshold"]:

            return self.predict_one(
                x,
                node["left"]
            )

        else:

            return self.predict_one(
                x,
                node["right"]
            )


    # ======================================================
    # Predict
    # ======================================================

    def predict(self, X):

        predictions = []

        for x in X:

            predictions.append(
                self.predict_one(
                    x,
                    self.tree
                )
            )

        return np.array(predictions)


# ==========================================================
# Execution Example
# ==========================================================

if __name__ == "__main__":

    # ------------------------------------------------------
    # Create X
    # ------------------------------------------------------

    X = np.array([
        [1, 2],
        [2, 3],
        [2, 1],
        [3, 2],
        [8, 8],
        [9, 8],
        [8, 9],
        [9, 9]
    ])


    # ------------------------------------------------------
    # Create y
    # ------------------------------------------------------

    y = np.array([
        0,
        0,
        0,
        0,
        1,
        1,
        1,
        1
    ])


    # ------------------------------------------------------
    # Train Model
    # ------------------------------------------------------

    model = DecisionTree(
        max_depth=3
    )

    model.fit(
        X,
        y
    )


    # ------------------------------------------------------
    # Prediction
    # ------------------------------------------------------

    predictions = model.predict(
        X
    )


    # ------------------------------------------------------
    # Print Results
    # ------------------------------------------------------

    print("Predictions:")
    print(predictions)

    print("\nActual:")
    print(y)

In [ ]:
import numpy as np


# ==========================================================
# Neural Network
# ==========================================================

class NeuralNetwork:

    def __init__(
        self,
        input_size,
        hidden_size,
        output_size,
        lr=0.01,
        epochs=100
    ):

        self.lr = lr
        self.epochs = epochs

        # --------------------------------------------------
        # Initialize parameters
        # --------------------------------------------------

        self.W1 = np.random.randn(
            input_size,
            hidden_size
        )

        self.b1 = np.zeros(
            (1, hidden_size)
        )

        self.W2 = np.random.randn(
            hidden_size,
            output_size
        )

        self.b2 = np.zeros(
            (1, output_size)
        )


    # ======================================================
    # ReLU
    # ======================================================

    def relu(self, X):

        return np.maximum(
            0,
            X
        )


    # ======================================================
    # Softmax
    # ======================================================

    def softmax(self, X):

        # Numerical stability

        X = X - np.max(
            X,
            axis=1,
            keepdims=True
        )

        exp_X = np.exp(X)

        return exp_X / np.sum(
            exp_X,
            axis=1,
            keepdims=True
        )


    # ======================================================
    # Cross Entropy Loss
    # ======================================================

    def loss(self, y, y_pred):

        eps = 1e-12

        return -np.mean(
            np.sum(
                y * np.log(
                    y_pred + eps
                ),
                axis=1
            )
        )


    # ======================================================
    # Forward Pass
    # ======================================================

    def forward(self, X):

        # --------------------------------------------------
        # Layer 1
        # --------------------------------------------------

        Z1 = X @ self.W1 + self.b1

        A1 = self.relu(Z1)


        # --------------------------------------------------
        # Layer 2
        # --------------------------------------------------

        Z2 = A1 @ self.W2 + self.b2

        A2 = self.softmax(Z2)


        # --------------------------------------------------
        # Return values needed for backpropagation
        # --------------------------------------------------

        return Z1, A1, A2


    # ======================================================
    # Backpropagation
    # ======================================================

    def backward(
        self,
        X,
        y,
        Z1,
        A1,
        y_pred
    ):

        n_samples = X.shape[0]


        # --------------------------------------------------
        # Output layer gradients
        # --------------------------------------------------

        dZ2 = y_pred - y

        dW2 = (
            1 / n_samples
        ) * (
            A1.T @ dZ2
        )

        db2 = (
            1 / n_samples
        ) * np.sum(
            dZ2,
            axis=0,
            keepdims=True
        )


        # --------------------------------------------------
        # Hidden layer gradients
        # --------------------------------------------------

        dA1 = dZ2 @ self.W2.T

        # ReLU derivative

        dZ1 = dA1 * (
            Z1 > 0
        )


        dW1 = (
            1 / n_samples
        ) * (
            X.T @ dZ1
        )

        db1 = (
            1 / n_samples
        ) * np.sum(
            dZ1,
            axis=0,
            keepdims=True
        )


        # --------------------------------------------------
        # Update parameters
        # --------------------------------------------------

        self.W1 -= self.lr * dW1

        self.b1 -= self.lr * db1

        self.W2 -= self.lr * dW2

        self.b2 -= self.lr * db2


    # ======================================================
    # Fit
    # ======================================================

    def fit(self, X, y):

        for epoch in range(
            self.epochs
        ):

            # --------------------------------------------------
            # Forward
            # --------------------------------------------------

            Z1, A1, y_pred = self.forward(
                X
            )


            # --------------------------------------------------
            # Loss
            # --------------------------------------------------

            loss = self.loss(
                y,
                y_pred
            )


            # --------------------------------------------------
            # Backward
            # --------------------------------------------------

            self.backward(
                X,
                y,
                Z1,
                A1,
                y_pred
            )


            if epoch % 10 == 0:

                print(
                    f"Epoch: {epoch} | Loss: {loss:.4f}"
                )


    # ======================================================
    # Predict
    # ======================================================

    def predict(self, X):

        # Run forward pass

        _, _, probabilities = self.forward(
            X
        )


        # Return class with highest probability

        return np.argmax(
            probabilities,
            axis=1
        )


# ==========================================================
# Execution Example
# ==========================================================

if __name__ == "__main__":

    np.random.seed(42)


    # ======================================================
    # Create X
    # ======================================================

    # 100 examples
    # 4 input features

    X = np.random.randn(
        100,
        4
    )


    # ======================================================
    # Create y
    # ======================================================

    # 3 classes

    labels = np.random.randint(
        0,
        3,
        size=100
    )


    # One-hot encoding

    y = np.zeros(
        (100, 3)
    )

    y[
        np.arange(100),
        labels
    ] = 1


    # ======================================================
    # Create Model
    # ======================================================

    model = NeuralNetwork(
        input_size=4,
        hidden_size=8,
        output_size=3,
        lr=0.01,
        epochs=100
    )


    # ======================================================
    # Train Model
    # ======================================================

    model.fit(
        X,
        y
    )


    # ======================================================
    # Prediction
    # ======================================================

    predictions = model.predict(
        X
    )


    # ======================================================
    # Print
    # ======================================================

    print("\nPredictions:")

    print(
        predictions[:10]
    )